# Machine Learning sobre Gasto Publico MEF

**Fase 6 del pipeline v2**

Dos problemas de ML sobre los datos de ejecucion presupuestal:

1. **Regresion**: Predecir el monto de ejecucion (`MONTO_DEVENGADO`) a partir
   del departamento, sector, nivel de gobierno y mes.
2. **Clustering**: Agrupar jurisdicciones por patrones de gasto similares
   para detectar perfiles de ejecucion.

Basado en las conclusiones del EDA (notebook 01).

## 1. Setup y carga de datos

In [ ]:
import os
import sys

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from config.logging_config import setup_logging
from config.settings import settings
from config.spark_config import crear_spark_session

setup_logging()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# Cargar datos
spark = crear_spark_session(app_name="ml_fase6")

from src.calidad.filtros_uat import aplicar_filtros_uat
from src.ingesta.lector_csv import leer_csv_con_schema

RUTA_CSV = settings.data.ruta_csv_raw
RUTA_SAMPLE = os.path.join(ROOT, "tests", "data", "sample_gasto.csv")
ruta = str(RUTA_CSV) if RUTA_CSV.exists() else RUTA_SAMPLE

df_spark = leer_csv_con_schema(spark, ruta)
df_limpio, _ = aplicar_filtros_uat(df_spark, settings.calidad)
df = df_limpio.toPandas()

print(f"Dataset: {len(df):,} registros, {len(df.columns)} columnas")
df.head()

## 2. Feature Engineering

Convertir variables categoricas (departamento, sector, etc.) a numericas
para que los modelos puedan procesarlas.

In [ ]:
from src.ml.features import preparar_features_clustering, preparar_features_registro

# Features para regresion (por registro individual)
X, y, nombres = preparar_features_registro(df)

print(f"Features: {X.shape[1]}")
print(f"Registros: {X.shape[0]:,}")
print(f"Target (media): {y.mean():,.2f}")
print(f"\nPrimeras 10 features: {nombres[:10]}")

## 3. Regresion — Predecir monto de ejecucion

Tres modelos comparados: Linear Regression, Random Forest, Gradient Boosting.
Evaluacion con train/test split (80/20) y cross-validation (k=5).

In [ ]:
from src.ml.evaluacion import guardar_resultados, serializar_modelo, tabla_comparativa
from src.ml.sklearn_models import entrenar_regresores

resultados_reg = entrenar_regresores(X, y)

# Tabla comparativa
print("\nComparativa de modelos de regresion")
print("=" * 68)
print(tabla_comparativa(resultados_reg))

In [ ]:
# Visualizar R2 de cross-validation por modelo
fig, ax = plt.subplots(figsize=(10, 5))

nombres_modelos = [r["nombre"] for r in resultados_reg]
cv_means = [r["metricas_cv"]["r2_mean"] for r in resultados_reg]
cv_stds = [r["metricas_cv"]["r2_std"] for r in resultados_reg]

colors = ["steelblue", "seagreen", "coral"]
bars = ax.bar(nombres_modelos, cv_means, yerr=cv_stds, capsize=5,
              color=colors, edgecolor="black", alpha=0.8)
ax.set_ylabel("R2 (Cross-Validation)")
ax.set_title("Comparativa de modelos — R2 con cross-validation (k=5)")
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)

for bar, mean in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{mean:.4f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

## 4. Feature Importance

Que variables influyen mas en la prediccion? Esto nos dice donde
buscar si queremos mejorar la precision del modelo.

In [ ]:
# Tomar el mejor modelo con feature_importance
mejor = max(
    [r for r in resultados_reg if r["feature_importance"]],
    key=lambda r: r["metricas_cv"]["r2_mean"],
)

importance = mejor["feature_importance"]
top_features = dict(list(importance.items())[:15])

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(list(top_features.keys())[::-1], list(top_features.values())[::-1],
        color="steelblue", edgecolor="black", alpha=0.8)
ax.set_xlabel("Importancia")
ax.set_title(f"Top 15 features — {mejor['nombre']}")
plt.tight_layout()
plt.show()

print(f"\nModelo: {mejor['nombre']}")
for feat, imp in list(top_features.items())[:10]:
    print(f"  {feat:.<40s} {imp:.4f}")

## 5. Clustering — Agrupar jurisdicciones

Agrupamos por (departamento, sector) y buscamos patrones:
grupos de jurisdicciones con perfiles de gasto similares.

In [ ]:
from src.ml.evaluacion import guardar_resultados_clustering
from src.ml.sklearn_models import entrenar_clustering

# Preparar features agregadas
df_feat_clust, df_info_clust = preparar_features_clustering(df)
print(f"Grupos para clustering: {len(df_feat_clust)}")
print(f"Features por grupo: {df_feat_clust.shape[1]}")

# Entrenar
if len(df_feat_clust) >= 4:  # necesitamos al menos 4 grupos para clustering
    resultado_clust = entrenar_clustering(df_feat_clust, df_info_clust)

    print(f"\nK optimo (KMeans): {resultado_clust['k_optimo']}")
    print(f"Silhouette KMeans: {resultado_clust['silhouette_kmeans']:.4f}")
    print(f"Clusters DBSCAN: {resultado_clust['n_clusters_dbscan']}")
    print(f"Puntos ruido DBSCAN: {resultado_clust['n_ruido_dbscan']}")
    if resultado_clust['silhouette_dbscan']:
        print(f"Silhouette DBSCAN: {resultado_clust['silhouette_dbscan']:.4f}")
else:
    print("Pocos grupos para clustering. Usar el dataset completo.")
    resultado_clust = None

In [ ]:
# Elbow Method + Silhouette
if resultado_clust:
    elbow = resultado_clust["elbow_data"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(elbow["k_values"], elbow["inertias"], "bo-", markersize=6)
    axes[0].axvline(resultado_clust["k_optimo"], color="red", linestyle="--",
                    label=f"K optimo = {resultado_clust['k_optimo']}")
    axes[0].set_xlabel("Numero de clusters (K)")
    axes[0].set_ylabel("Inercia")
    axes[0].set_title("Elbow Method")
    axes[0].legend()

    axes[1].plot(elbow["k_values"], elbow["silhouettes"], "go-", markersize=6)
    axes[1].axvline(resultado_clust["k_optimo"], color="red", linestyle="--")
    axes[1].set_xlabel("Numero de clusters (K)")
    axes[1].set_ylabel("Silhouette Score")
    axes[1].set_title("Silhouette por K")

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualizar clusters (2D con las 2 features mas importantes)
if resultado_clust:
    from sklearn.decomposition import PCA

    scaler = resultado_clust["scaler"]
    X_scaled = scaler.transform(df_feat_clust)

    if X_scaled.shape[1] > 2:
        pca = PCA(n_components=2)
        X_2d = pca.fit_transform(X_scaled)
        xlabel = f"PC1 ({pca.explained_variance_ratio_[0]:.1%})"
        ylabel = f"PC2 ({pca.explained_variance_ratio_[1]:.1%})"
    else:
        X_2d = X_scaled[:, :2]
        xlabel, ylabel = df_feat_clust.columns[0], df_feat_clust.columns[1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # KMeans
    scatter = axes[0].scatter(X_2d[:, 0], X_2d[:, 1],
                             c=resultado_clust["labels_kmeans"],
                             cmap="Set2", edgecolors="black", s=60)
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel(ylabel)
    axes[0].set_title(f"KMeans (k={resultado_clust['k_optimo']})")
    plt.colorbar(scatter, ax=axes[0], label="Cluster")

    # DBSCAN
    scatter2 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1],
                              c=resultado_clust["labels_dbscan"],
                              cmap="Set2", edgecolors="black", s=60)
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel(ylabel)
    axes[1].set_title(f"DBSCAN ({resultado_clust['n_clusters_dbscan']} clusters)")
    plt.colorbar(scatter2, ax=axes[1], label="Cluster")

    plt.tight_layout()
    plt.show()

## 6. Guardar modelos y resultados

In [ ]:
# Guardar metricas en JSON (tracking local)
guardar_resultados(resultados_reg)
if resultado_clust:
    guardar_resultados_clustering(resultado_clust)

# Serializar el mejor modelo de regresion
mejor_reg = max(resultados_reg, key=lambda r: r["metricas_cv"]["r2_mean"])
serializar_modelo(mejor_reg["modelo"], f"mejor_regresor_{mejor_reg['nombre'].lower()}")

# Serializar clustering
if resultado_clust:
    serializar_modelo(resultado_clust["kmeans_modelo"], "kmeans_jurisdicciones")
    serializar_modelo(resultado_clust["scaler"], "scaler_clustering")

print("\nModelos y metricas guardados en models/")

## 7. Resumen

### Regresion
- Se evaluaron 3 modelos para predecir el monto de ejecucion presupuestal
- Las features categoricas (departamento, sector, nivel de gobierno, mes) se
  convirtieron a variables dummy con one-hot encoding
- Cada modelo se evaluo con train/test split y cross-validation

### Clustering
- Se agruparon jurisdicciones (departamento x sector) por patrones de gasto
- Features agregadas: total, promedio, mediana, desviacion, conteo, CV, rango
- KMeans con Elbow Method + DBSCAN como alternativa no parametrica
- Silhouette Score para evaluar la calidad de los clusters

### Para Fase 7 (MLlib)
- Replicar estos modelos con PySpark MLlib para comparar rendimiento
- Usar el pipeline: StringIndexer -> OneHotEncoder -> VectorAssembler -> Modelo

In [ ]:
spark.stop()
print("Sesion cerrada. Fase 6 completada.")